# SemMap — Shared Shape Dictionary / nSBM / Huffman · T4/L4 Colab

**Запуск ConceptNet-10k → 100k**, единый словарь графовых форм между уровнями с точным декодированием. По умолчанию тестируется `ISOMORPHISM_LEVEL="recursive"`; уровень сравнения не ослабляет exact identity.

**Ресурсы:** graph-tool nSBM — CPU/OpenMP, VF2 — ограниченные CPU-процессы, извлечение следующей пачки перекрывается с VF2. T4/L4 через CuPy применяется к степеням, W-проекции и Wishart-kNN. **nSBM/VF2 не исполняются на GPU.**

Данные и вычисления на локальном SSD `/content`, подтверждённые архивы и возобновляемые scan shards — на Google Drive. На каждом шаге выводится **общий прогресс по запланированным фазам** и **счётчики обработанных центров/групп**. После успешного smoke с 10k можно выбрать 100k (High-RAM рекомендуется).

Старые v1-запуски нельзя возобновлять с новым форматом: новый `RUN_ID` и `RUN_MODE="NEW"`. Для возобновления одного и того же нового запуска не менять Git SHA, параметры и входной файл.

**Важно:** журнал запуска создаётся вне каталога `RUN_LOCAL`, чтобы строгая проверка `NEW` не считала каталог занятым. Для исправленной версии по умолчанию выбран новый `RUN_ID` (`-v2`); прежние файлы не удаляются.


In [ ]:
# 1. Единственная конфигурационная ячейка
from pathlib import Path
REPO_URL = "https://github.com/SemanticMap/semgraphex.git"
REPO_BRANCH = "feature/shared-shapes-colab-t4l4"
SOURCE_REVISION = "f6c9bb350d072ee7b64f6823964476dcd02307bf"
PROFILE = "10k"  # "10k" | "100k"
RUN_MODE = "NEW"  # "NEW" | "RESUME"; не меняйте при изменении кода/данных без нового RUN_ID
ISOMORPHISM_LEVEL = "recursive"  # topology | directed | typed | recursive
RUN_ID = f"nsbm-shared-shapes-{PROFILE}-{ISOMORPHISM_LEVEL}-v2"
SEED = 42
CPU_BUDGET = 12   # верхняя граница, фактическая учитывает cgroup/affinity
RESTARTS = 3
METHODS = "nsbm,plain,shuffled,wishart,nsbm_wishart" if PROFILE == "10k" else "nsbm,shuffled,wishart"
GPU_REQUIRED = True  # T4/L4, False только для CPU-проверки
DISCOVERY = None  # профиль: 800 для 10k, 3000 для 100k
MAX_EGO_NODES = 7
MIN_FIGURE_NODES = 3
MIN_SUPPORT = 3
MAX_RECURSIVE_LEVELS = 3
SYNC_INTERVAL_SECONDS = 120
DRIVE_PROJECT = Path("/content/drive/MyDrive/SemanticMap")
INPUT_PATHS = {
    "10k": DRIVE_PROJECT / "colab/hypothesis/inputs/h1-cn10k-typed-full-01/conceptnet_en_10k_relations.tsv",
    "100k": DRIVE_PROJECT / "colab/hypothesis/inputs/conceptnet_en_100k_relations.tsv",
}
assert PROFILE in ("10k", "100k") and RUN_MODE in ("NEW", "RESUME")
assert ISOMORPHISM_LEVEL in ("topology", "directed", "typed", "recursive")

In [ ]:
# 2. Ресурсы и реальный CPU-бюджет
import os, sys, shutil, subprocess, json, hashlib, time

def allowed_cpus():
    n = os.cpu_count() or 1
    if hasattr(os, "sched_getaffinity"):
        n = min(n, len(os.sched_getaffinity(0)))
    quota_file = Path("/sys/fs/cgroup/cpu.max")
    if quota_file.exists():
        quota, period = quota_file.read_text().split()
        if quota != "max":
            n = min(n, max(1, (int(quota) + int(period) - 1)//int(period)))
    return max(1, n)

CPU_EFFECTIVE = min(CPU_BUDGET, allowed_cpus())
gpu = (subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
       if shutil.which("nvidia-smi") else subprocess.CompletedProcess([], 1, "", ""))
GPU_VISIBLE = gpu.returncode == 0 and "GPU" in gpu.stdout
if GPU_REQUIRED and not GPU_VISIBLE:
    raise RuntimeError("Нет GPU. В Colab выберите Runtime → Change runtime type → T4/L4 GPU.")
DEVICE = "cuda" if GPU_REQUIRED else "auto"
free_gb = shutil.disk_usage("/content").free / 2**30
print({"python": sys.version.split()[0], "cpu_available": allowed_cpus(),
       "cpu_budget": CPU_EFFECTIVE, "free_disk_GiB": round(free_gb,1),
       "gpu": gpu.stdout.strip() if GPU_VISIBLE else None})
if free_gb < 7:
    raise RuntimeError("Для micromamba/graph-tool нужно около 7 GiB свободного диска")

In [ ]:
# 3. Drive: длительное хранение, но не вычисление
from google.colab import drive
drive.mount("/content/drive")
PERSISTENT = DRIVE_PROJECT / "colab" / "hypothesis" / "runs" / RUN_ID
SCRATCH = Path("/content/semmap_nsbm_huffman")
REPO = Path("/content/semmap_nsbm_huffman_repo")
ENV_ROOT = Path("/content/semmap_nsbm_huffman_env")
BIN_ROOT = Path("/content/semmap_nsbm_huffman_bin")
RUN_LOCAL = SCRATCH / "runs" / RUN_ID
SCRATCH.mkdir(parents=True, exist_ok=True)
print("Run:", RUN_ID, "\nDrive:", PERSISTENT)

In [ ]:
# 4. Воспроизводимый checkout ровно одной ревизии
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "75", "--branch", REPO_BRANCH,
                    REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "75",
                    "origin", REPO_BRANCH], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_REVISION], check=True)
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert HEAD == SOURCE_REVISION
print("PINNED:", HEAD)

In [ ]:
# 5. graph-tool + CuPy/CUDA12 в одном conda-forge окружении
# Проверяем реальное умножение на GPU, а не только наличие устройства.
BIN_ROOT.mkdir(parents=True, exist_ok=True)
MAMBA = BIN_ROOT / "bin" / "micromamba"
ENV_PY = ENV_ROOT / "bin" / "python"
if not MAMBA.exists():
    subprocess.run(["bash", "-lc", "set -euo pipefail; curl -fsSL "
      "https://micro.mamba.pm/api/micromamba/linux-64/latest | "
      f"tar -xj -C {BIN_ROOT} bin/micromamba"], check=True)

mm_env = dict(os.environ, MAMBA_ROOT_PREFIX="/content/semmap_nsbm_huffman_mamba")
packages = ["python=3.12", "graph-tool-base=2.98", "numpy", "scipy",
            "scikit-learn", "networkx", "pyyaml", "psutil",
            "threadpoolctl", "pytest", "pip"]
if GPU_REQUIRED:
    # conda-forge cupy brings libcublas, CUDA runtime and NVRTC as dependencies.
    # cuda-version=12 alone is only a version selector, not the CUDA libraries.
    packages += ["cupy", "cuda-version=12"]
if not ENV_PY.is_file():
    subprocess.run([str(MAMBA), "create", "-y", "-p", str(ENV_ROOT),
                    "-c", "conda-forge", "--strict-channel-priority",
                    *packages], check=True, env=mm_env)
subprocess.run([str(ENV_PY), "-m", "pip", "install", "--no-deps", "-e",
                str(REPO)], check=True)

# This environment is also passed to the experiment subprocess (cell 8).
# With micromamba, ENV_ROOT/lib is the location of conda CUDA libraries.
GPU_SUBPROCESS_ENV = dict(os.environ)
GPU_SUBPROCESS_ENV["LD_LIBRARY_PATH"] = (
    str(ENV_ROOT / "lib") +
    (":" + os.environ["LD_LIBRARY_PATH"] if os.environ.get("LD_LIBRARY_PATH") else "")
)
CUDA_PROBE = """
import cupy as cp
import numpy as np
assert cp.cuda.runtime.getDeviceCount() > 0
a = cp.asarray([[1.0, 2.0], [3.0, 4.0]], dtype=cp.float32)
result = a @ a  # deliberately loads cuBLAS (libcublas.so.12)
assert np.allclose(cp.asnumpy(result), [[7., 10.], [15., 22.]])
assert np.isclose(float(cp.asnumpy(cp.sum(cp.square(a)))), 30.0)
cp.cuda.Stream.null.synchronize()
print("CuPy GPU matrix multiply OK:", cp.cuda.runtime.getDeviceProperties(0)["name"])
"""
if GPU_REQUIRED:
    def probe_cuda():
        return subprocess.run([str(ENV_PY), "-c", CUDA_PROBE],
                              env=GPU_SUBPROCESS_ENV, capture_output=True, text=True)
    probe = probe_cuda()
    if probe.returncode != 0:
        print("CUDA/cuBLAS preflight failed; repairing GPU dependencies in existing conda env.")
        print(probe.stderr[-1200:])
        # Prior notebook installed a pip-only wheel without runtime components.
        # Remove it before installing conda-forge CuPy to avoid mixed packages.
        old_wheel = subprocess.run([str(ENV_PY), "-m", "pip", "show", "cupy-cuda12x"],
                                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        if old_wheel.returncode == 0:
            subprocess.run([str(ENV_PY), "-m", "pip", "uninstall", "-y",
                            "cupy-cuda12x"], check=True)
        subprocess.run([str(MAMBA), "install", "-y", "-p", str(ENV_ROOT),
                        "-c", "conda-forge", "--strict-channel-priority",
                        "cupy", "cuda-version=12"], check=True, env=mm_env)
        probe = probe_cuda()
    if probe.returncode != 0:
        raise RuntimeError(
            "GPU preflight: cuBLAS matrix multiplication failed even after CUDA "
            "dependency installation. Inspect driver / CUDA compatibility.\n" +
            probe.stderr[-2500:]
        )
    print(probe.stdout.strip())

subprocess.run([str(ENV_PY), "-c",
    "import graph_tool.all as gt; print('graph-tool:', gt.__version__)"],
    check=True, env=GPU_SUBPROCESS_ENV)

In [ ]:
# 6. Быстрый smoke (без настоящего обучения nSBM)
check_env = dict(GPU_SUBPROCESS_ENV, OPENBLAS_NUM_THREADS="1", MKL_NUM_THREADS="1",
                 NUMEXPR_NUM_THREADS="1", PYTHONHASHSEED="0",
                 PYTHONWARNINGS="ignore:The hashes produced for directed graphs changed in version v3.5:UserWarning")
subprocess.run([str(ENV_PY), "-m", "pytest", "-q",
    "tests/test_grammar_huffman.py",
    "tests/test_nsbm_shape_identity.py", "tests/test_nsbm_shared_hierarchy.py",
    "tests/test_nsbm_colab_pipeline.py", "tests/test_nsbm_huffman_experiment.py",
    "tests/test_nsbm_checkpoint_independence.py", "tests/test_nsbm_recursive_codec.py",
    "tests/test_nsbm_measured_selection.py", "tests/test_nsbm_fit_checkpoint.py",
    "tests/test_h1_runner.py"],
    cwd=REPO, env=check_env, check=True)

In [ ]:
# 7. Подготовка входа в локальном SSD (никогда не считаем из Drive)
INPUT = Path(INPUT_PATHS[PROFILE])
if not INPUT.is_file():
    raise FileNotFoundError(f"Не найден подготовленный {PROFILE} ConceptNet TSV: {INPUT}. "
        "Укажите путь в INPUT_PATHS. Для 100k нужен репрезентативно отобранный 100k TSV, "
        "не первые 100k строк большого дампа.")
STAGED = SCRATCH / f"input_{PROFILE}.tsv"
if not STAGED.is_file() or STAGED.stat().st_size != INPUT.stat().st_size:
    shutil.copy2(INPUT, STAGED)

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024*1024), b""):
            digest.update(block)
    return digest.hexdigest()
INPUT_HASH = file_sha256(STAGED)
if file_sha256(INPUT) != INPUT_HASH:
    raise RuntimeError("Drive/local input checksum differs")
print("Input staged:", STAGED, "size MiB:", round(STAGED.stat().st_size/2**20,1),
      "SHA256:", INPUT_HASH)
if RUN_MODE == "RESUME":
    if not PERSISTENT.is_dir():
        raise FileNotFoundError(f"No persisted checkpoint: {PERSISTENT}")
    shutil.copytree(PERSISTENT, RUN_LOCAL, dirs_exist_ok=True)
else:
    if RUN_LOCAL.exists() and any(RUN_LOCAL.iterdir()):
        raise RuntimeError("NEW требует пустую локальную папку; выберите новый RUN_ID")
    if PERSISTENT.exists() and any(PERSISTENT.iterdir()):
        raise RuntimeError("NEW не должен перезаписывать существующий Drive run")

In [ ]:
# 8. Запуск, живой прогресс, SHA-verified atomic checkpoint → Drive
# В Drive публикуются только принятые рекурсивные ZIP (имеющие SHA-метаданные).
import re

def sync_checkpoint(final=False):
    # The CLI owns NEW initialization. A failed preflight must not create a
    # phantom run on Drive with only a log and no resumable manifest.
    if not (RUN_LOCAL / "manifest.json").is_file():
        return 0
    PERSISTENT.mkdir(parents=True, exist_ok=True)
    synced = 0
    sources = list(RUN_LOCAL.rglob("*"))
    if LOG.is_file():
        sources.append(LOG)
    for source in sorted(sources):
        if not source.is_file() or source.name == "COMPLETED" or source.name.endswith(".tmp"):
            continue
        if source == RUN_LOCAL / "execution.log":
            continue  # stale log restored from Drive on RESUME; current log is LOG
        relative = (Path("execution.log") if source == LOG
                    else source.relative_to(RUN_LOCAL))
        if source.name.startswith("transition_") and source.suffix == ".zip":
            continue  # temporary transition; the accepted level embeds its delta
        if source.parent == RUN_LOCAL / "recursive" and re.fullmatch(r"level_\d{3}\.zip", source.name):
            metadata = source.with_suffix(".json")
            if not metadata.is_file():
                continue  # candidate is not accepted/published yet
            try:
                expected = json.loads(metadata.read_text())["sha256"]
                if file_sha256(source) != expected:
                    continue
            except (KeyError, OSError, ValueError):
                continue
        dest = PERSISTENT / relative
        dest.parent.mkdir(parents=True, exist_ok=True)
        if (dest.is_file() and dest.stat().st_size == source.stat().st_size
                and dest.stat().st_mtime_ns == source.stat().st_mtime_ns):
            # A same-size corrupted ZIP must not silently count as synced.
            if source.suffix != ".zip" or file_sha256(dest) == file_sha256(source):
                continue
        temporary = dest.with_name(dest.name + ".sync_tmp")
        try:
            shutil.copy2(source, temporary)
            copied_hash = file_sha256(temporary)
            if copied_hash != file_sha256(source):
                temporary.unlink(missing_ok=True)  # changing source; retry next sync
                continue
            os.replace(temporary, dest)
            synced += 1
        except OSError:
            temporary.unlink(missing_ok=True)
            raise

    if final:
        assert (RUN_LOCAL / "COMPLETED").is_file()
        report = json.loads((RUN_LOCAL / "report.json").read_text())
        for method, record in report["methods"].items():
            for coding, artifact in record["coding"].items():
                for stem, checksum in (("grammar", "archive_sha256"), ("bundle", "bundle_sha256")):
                    name = (f"grammar_{method}_{coding}.zip" if stem == "grammar"
                            else f"bundle_{method}_{coding}.zip")
                    if file_sha256(PERSISTENT / name) != artifact[checksum]:
                        raise RuntimeError(f"Drive checksum mismatch: {name}")
        recursive = report.get("recursive")
        if recursive:
            local_archive = Path(recursive["archive"])
            relative = local_archive.relative_to(RUN_LOCAL)
            if file_sha256(PERSISTENT / relative) != recursive["archive_sha256"]:
                raise RuntimeError("Recursive archive was not verified on Drive")
        # COMPLETED is the LAST published marker.
        shutil.copy2(RUN_LOCAL / "COMPLETED", PERSISTENT / "COMPLETED")
    return synced

cmd = [str(ENV_PY), "-m", "semmap_haken.hypothesis.grammar_experiment",
       "--input", str(STAGED), "--output", str(RUN_LOCAL),
       "--profile", PROFILE, "--mode", RUN_MODE, "--seed", str(SEED),
       "--restarts", str(RESTARTS), "--cpu-budget", str(CPU_EFFECTIVE),
       "--device", DEVICE, "--methods", METHODS,
       "--isomorphism-level", ISOMORPHISM_LEVEL,
       "--max-ego-nodes", str(MAX_EGO_NODES),
       "--min-figure-nodes", str(MIN_FIGURE_NODES),
       "--min-support", str(MIN_SUPPORT),
       "--max-recursive-levels", str(MAX_RECURSIVE_LEVELS)]
if DISCOVERY is not None:
    cmd += ["--discovery", str(DISCOVERY)]

LOG = SCRATCH / f"{RUN_ID}.log"  # outside RUN_LOCAL so CLI NEW sees an empty directory
print("Execution log (local SSD):", LOG)
run_env = dict(check_env, PYTHONUNBUFFERED="1", OMP_NUM_THREADS="1")
def print_updates(previous_offset):
    with LOG.open("r", encoding="utf-8", errors="replace") as stream:
        stream.seek(previous_offset)
        new_lines = stream.readlines()
        previous_offset = stream.tell()
    for raw in new_lines:
        if raw.startswith("PROGRESS "):
            event = json.loads(raw[len("PROGRESS "):])
            print(f"{event['overall_percent']:6.2f}% всего | "
                  f"шаг {event['step']}/{event['steps_total']} | {event['phase']} | "
                  f"{event['completed']}/{event['total']} "
                  f"({event['phase_percent']}%) | {event['elapsed_seconds']} сек", flush=True)
    return previous_offset

last_sync = time.monotonic()
position = 0
with LOG.open("w", encoding="utf-8") as log:
    proc = subprocess.Popen(cmd, cwd=REPO, env=run_env,
                            stdout=log, stderr=subprocess.STDOUT)
    try:
        while proc.poll() is None:
            time.sleep(3)
            log.flush()
            position = print_updates(position)
            if time.monotonic() - last_sync >= SYNC_INTERVAL_SECONDS:
                nfiles = sync_checkpoint()
                last_sync = time.monotonic()
                print("Drive checkpoint:", nfiles, "новых/изменённых файлов", flush=True)
        log.flush()
        position = print_updates(position)
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        raise
    finally:
        # If the process failed, still save verified partial state for RESUME.
        sync_checkpoint(final=proc.poll() == 0)

if proc.returncode != 0:
    print("\n".join(LOG.read_text(errors="replace").splitlines()[-55:]))
    raise RuntimeError(f"Experiment exited with code {proc.returncode}")
assert (PERSISTENT / "COMPLETED").is_file()
print("DONE", PERSISTENT)


In [ ]:
# 9. Итог: фактический размер архивов (а не nSBM MDL)
report = json.loads((PERSISTENT/"report.json").read_text(encoding="utf-8"))
print("ConceptNet:", report["counts"])
print("CPU:", report["cpu_budget"], "| GPU:", report["degree_backend"],
      "| W:", report["projection"])
print("Baseline bundle:", report["baseline"]["full_bundle_bytes"], "bytes")
for method, row in report["methods"].items():
    print(method, "selected:", row["selection"]["accepted_figures"])
    for coding, ratio in report["ratios_full_bundle"][method].items():
        print("  ", coding, "R =", round(ratio, 4),
              "full ZIP:", row["coding"][coding]["full_bundle_bytes"])
print("Report:", PERSISTENT/"report.json")
print("COMPLETED:", (PERSISTENT/"COMPLETED").exists())
if report.get("recursive"):
    rec = report["recursive"]
    print("Recursive hierarchy:", rec["accepted_levels"], "levels;",
          rec["archive_bytes"], "bytes; status:", rec["stop_reason"])
    print("Recursive archive:", rec["archive"])

print("Stage timing (seconds):", json.dumps(report.get("phase_timing_seconds", {}), ensure_ascii=False, indent=2))
for method, row in report["methods"].items():
    print(method, "timing:", row.get("phase_timing_seconds", {}),
          "scan:", row["search"]["scan_checkpoint"].get("centers_processed"), "/",
          row["search"]["scan_checkpoint"].get("centers_total"))
if report.get("recursive"):
    for stage in report["recursive"]["attempted"]:
        print("Recursive L", stage["scale"], "timing:", stage.get("phase_timing_seconds"),
              "scan:", stage.get("scan", {}).get("centers_processed"), "/",
              stage.get("scan", {}).get("centers_total"))


### Контроль качества результатов

- Сначала `10k`: правильный GPU preflight, пройденные smoke-тесты, `PROGRESS` от 0 до 100%, `report.json` и `COMPLETED` на Drive.
- Затем `100k` с другим `RUN_ID`, желательно High-RAM; nSBM/VF2 CPU, GPU участвует только в отмеченных CuPy-этапах.
- `RESUME` — прежние данные, `SOURCE_REVISION` и параметры. Scan shards со SHA переиспользуются, неполные переделываются.
- `phase_timing_seconds` и `scan` позволяют оценить выигрыш перекрытия extraction/VF2. Время суммарных worker-фаз может перекрываться, поэтому не обязано суммироваться в wall-time.
- Сравнение объёмов — только по реальным байтам ZIP, а не по MDL. Без реального прогона на T4/L4 нельзя утверждать величину ускорения.
